In [126]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [127]:
delivery = pd.read_csv("../data/processed/Delivery.csv")
pickup = pd.read_csv("../data/processed/Pickup.csv")
grid = pd.read_csv("../data/processed/GridMovement.csv")
hubs = pd.read_csv("../data/processed/Hubs.csv")

# clean index columns
delivery = delivery.drop(columns=["Unnamed: 0"], errors="ignore")
pickup = pickup.drop(columns=["Unnamed: 0"], errors="ignore")


In [128]:
delivery["receipt_time"] = pd.to_datetime(delivery["receipt_time"])
delivery["sign_time"] = pd.to_datetime(delivery["sign_time"])

delivery["eta_minutes"] = (
    (delivery["sign_time"] - delivery["receipt_time"])
    .dt.total_seconds() / 60
)

# clean
delivery = delivery[(delivery["eta_minutes"] > 1) & (delivery["eta_minutes"] < 300)]

In [129]:
# time
delivery["hour"] = delivery["receipt_time"].dt.hour
delivery["weekday"] = delivery["receipt_time"].dt.weekday

# spatial
delivery["distance_km"] = np.sqrt(
    (delivery["receipt_lat"] - delivery["poi_lat"])**2 +
    (delivery["receipt_lng"] - delivery["poi_lng"])**2
) / 1000

# interaction
delivery["distance_hour_interaction"] = delivery["distance_km"] * delivery["hour"]

In [130]:
courier_stats = delivery.groupby("delivery_user_id").agg(
    courier_avg_eta=("eta_minutes", "mean"),
    courier_order_count=("eta_minutes", "count"),
    avg_distance=("distance_km", "mean")
).reset_index()

courier_stats["historical_speed"] = (
    courier_stats["avg_distance"] / (courier_stats["courier_avg_eta"] + 1e-6)
)

# ✅ time-aware load
courier_daily = delivery.groupby(
    ["delivery_user_id", "ds"]
).size().reset_index(name="courier_daily_load")

courier_hourly = delivery.groupby(
    ["delivery_user_id", "ds", "hour"]
).size().reset_index(name="courier_hourly_load")


In [131]:
aoi_stats = delivery.groupby("aoi_id").agg(
    aoi_mean_eta=("eta_minutes", "mean"),
    aoi_count=("eta_minutes", "count"),
    aoi_avg_distance=("distance_km", "mean")
).reset_index()

delivery = delivery.merge(aoi_stats, on="aoi_id", how="left")


In [132]:
# create grid key
grid["grid_key"] = grid["grid_x"].astype(str) + "_" + grid["grid_y"].astype(str)

grid_stats = grid.groupby("grid_key").agg(
    grid_avg_speed=("speed_kmph", "mean"),
    grid_total_activity=("distance_km", "sum")
).reset_index()

# map delivery to grid
delivery["grid_x"] = pd.qcut(delivery["poi_lng"], 20, labels=False)
delivery["grid_y"] = pd.qcut(delivery["poi_lat"], 20, labels=False)

delivery["grid_key"] = delivery["grid_x"].astype(str) + "_" + delivery["grid_y"].astype(str)

delivery = delivery.merge(grid_stats, on="grid_key", how="left")

In [133]:
delivery["nearest_hub_id"] = 0  # placeholder or from precomputed mapping

In [134]:
train = delivery[delivery["ds"] < 329]
test = delivery[delivery["ds"] >= 329]


In [135]:
target = "eta_minutes"

X_train = train.drop(columns=[target])
y_train = train[target]

X_test = test.drop(columns=[target])
y_test = test[target]


In [136]:
categorical_cols = [
    "aoi_id",
    "delivery_user_id",
    "from_dipan_id",
    "typecode",
    "city_name"
]

for col in categorical_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

In [137]:
model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

# drop identifier / datetime columns that are not features
drop_cols = ['order_id','receipt_time','sign_time','got_time','accept_time','book_start_time',
"receipt_lat", "receipt_lng", "pickup_hour", "pickup_to_delivery_km", "grid_x","grid_y","city_name","typecode",
    "avg_distance",
    "aoi_avg_distance",
]
X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns], errors='ignore')
X_test = X_test.drop(columns=[c for c in drop_cols if c in X_test.columns], errors='ignore')

# drop any remaining datetime cols
dt_cols = X_train.select_dtypes(include=['datetime64[ns]','datetime64']).columns.tolist()
X_train = X_train.drop(columns=dt_cols, errors='ignore')
X_test = X_test.drop(columns=[c for c in dt_cols if c in X_test.columns], errors='ignore')

# drop object/string cols that are not explicitly treated as categorical
obj_cols = [c for c in X_train.select_dtypes(include=['object','string']).columns if c not in categorical_cols]
X_train = X_train.drop(columns=obj_cols, errors='ignore')
X_test = X_test.drop(columns=[c for c in obj_cols if c in X_test.columns], errors='ignore')

# align train/test columns
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# fit model with valid categorical features
valid_cats = [c for c in categorical_cols if c in X_train.columns]
model.fit(X_train, y_train, categorical_feature=valid_cats)

preds = model.predict(X_test)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016163 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14015
[LightGBM] [Info] Number of data points in the train set: 313515, number of used features: 12
[LightGBM] [Info] Start training from score 101.094040


In [138]:
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

abs_error = np.abs(y_test - preds)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("P50:", np.percentile(abs_error, 50))
print("P90:", np.percentile(abs_error, 90))


MAE: 33.98501850141037
RMSE: 46.819054869280954
R2: 0.4825459111739896
P50: 24.724042944173284
P90: 75.4556434919073


In [139]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance.head(20))

,feature,importance
4,aoi_id,3202
1,delivery_user_id,2979
6,hour,747
10,aoi_mean_eta,546
0,from_dipan_id,294
8,distance_km,253
9,distance_hour_interaction,243
5,ds,218
11,aoi_count,164
2,poi_lng,132


In [140]:
final_features = [
    # IDs (strongest)
    "aoi_id",
    "delivery_user_id",
    "from_dipan_id",

    # Time
    "hour",
    "weekday",
    "ds",

    # Distance
    "distance_km",
    "distance_hour_interaction",

    # Courier (VERY IMPORTANT)
    "courier_hourly_load",
    "courier_daily_load",
    "courier_avg_eta",
    "courier_order_count",

    # Area
    "aoi_mean_eta",
    "aoi_count",

    # Location (region signal)
    "poi_lat",
    "poi_lng"
]


In [ ]:
# ---------------------------
# RENAME COLUMNS SAFELY
# ---------------------------
rename_map = { #because of merges, some columns have _x suffix - we want to rename them back to original names if they exist
    "aoi_mean_eta_x": "aoi_mean_eta",
    "aoi_count_x": "aoi_count",
    "aoi_avg_distance_x": "aoi_avg_distance",
}

for df in [train, test]:
    for old, new in rename_map.items():
        if old in df.columns:
            df.rename(columns={old: new}, inplace=True)


# ---------------------------
# CHECK FEATURES
# ---------------------------
missing = [col for col in final_features if col not in train.columns]
print("Missing features:", missing)


# ---------------------------
# MODEL DATA
# ---------------------------
# merge courier features before selecting final_features
train = train.merge(
    courier_stats[["delivery_user_id", "courier_avg_eta", "courier_order_count"]],
    on="delivery_user_id",
    how="left",
)
test = test.merge(
    courier_stats[["delivery_user_id", "courier_avg_eta", "courier_order_count"]],
    on="delivery_user_id",
    how="left",
)

train = train.merge(
    courier_daily,
    on=["delivery_user_id", "ds"],
    how="left",
)
test = test.merge(
    courier_daily,
    on=["delivery_user_id", "ds"],
    how="left",
)

train = train.merge(
    courier_hourly,
    on=["delivery_user_id", "ds", "hour"],
    how="left",
)
test = test.merge(
    courier_hourly,
    on=["delivery_user_id", "ds", "hour"],
    how="left",
)

train[["courier_hourly_load", "courier_daily_load", "courier_avg_eta", "courier_order_count"]] = (
    train[["courier_hourly_load", "courier_daily_load", "courier_avg_eta", "courier_order_count"]]
    .fillna(0)
)
test[["courier_hourly_load", "courier_daily_load", "courier_avg_eta", "courier_order_count"]] = (
    test[["courier_hourly_load", "courier_daily_load", "courier_avg_eta", "courier_order_count"]]
    .fillna(0)
)

X_train = train[final_features].copy()
X_test = test[final_features].copy()

y_train = train["eta_minutes"]
y_test = test["eta_minutes"]


# ---------------------------
# CATEGORICAL FEATURES
# ---------------------------
# align train/test columns before converting categorical features
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

categorical_cols = [
    "aoi_id",
    "delivery_user_id",
    "from_dipan_id"
]

cat_mappings = {}
for col in categorical_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        cat_mappings[col] = X_train[col].cat.categories.tolist()

        X_test[col] = X_test[col].astype("category")
        X_test[col] = X_test[col].cat.set_categories(X_train[col].cat.categories)

        X_train[col] = X_train[col].cat.add_categories(["missing"]).fillna("missing")
        X_test[col] = X_test[col].cat.add_categories(["missing"]).fillna("missing")


# ---------------------------
# MODEL
# ---------------------------
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)
preds = model.predict(X_test)


Missing features: ['courier_hourly_load', 'courier_daily_load', 'courier_avg_eta', 'courier_order_count']
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013708 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14626
[LightGBM] [Info] Number of data points in the train set: 313515, number of used features: 16
[LightGBM] [Info] Start training from score 101.094040


In [144]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

abs_err = np.abs(y_test - preds)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.3f}")
print(f"P50: {np.percentile(abs_err, 50):.2f}")
print(f"P90: {np.percentile(abs_err, 90):.2f}")

MAE: 32.68
RMSE: 45.57
R2: 0.510
P50: 23.29
P90: 72.71


In [145]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance.head(20))

,feature,importance
0,aoi_id,9865
1,delivery_user_id,9133
8,courier_hourly_load,1881
3,hour,1815
12,aoi_mean_eta,1436
2,from_dipan_id,1268
7,distance_hour_interaction,1065
6,distance_km,872
9,courier_daily_load,799
5,ds,688


In [ ]:
import pickle

pickle.dump(model, open("../models/lgbm_eta_model.pkl", "wb"))
pickle.dump(final_features, open("../models/features.pkl", "wb"))
pickle.dump(cat_mappings, open("../models/cat_mappings.pkl", "wb"))


In [147]:
courier_stats.to_pickle("../models/courier_stats.pkl")
aoi_stats.to_pickle("../models/aoi_stats.pkl")

In [148]:
courier_daily.to_pickle("../models/courier_daily.pkl")
courier_hourly.to_pickle("../models/courier_hourly.pkl")